# b3_agents

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): rule-agent behaviours as parameters. `B3_KNOBS` and `B3Params` (θ: `trustOne`, and the three behaviour-1 candidates `spread`, `shareAll` and `guess`); the `TrustOne`, `Spread` and `ShareAll` mixins, drawing from their own generator `b3Rng`; `b3Members`, which wraps a run's honest numpy members after `stage4Members` (and Phase 20's `bridgeMembers`) built them; and `b3Replay`, `colony_trace.replay`'s numpy branch with those members. Without `world['b3']`, or with every knob at its default, nothing changes and no generator is made (FR-A1 to FR-A3).

In [ ]:
#| default_exp b3_agents

In [ ]:
#| hide
from nbdev.showdoc import *

## Knobs and members

`trustOne` is a belief, not a coin per turn: the draw for (subtask, clue, value) is made the first time the agent sees that value leading with fewer than q senders, and kept for the board. `spread` retargets an open that the base agent chose (so `RuleBridge`'s relays and `GuardAgent`'s choices stay theirs); `shareAll` makes the share policy 'always' for one turn. `guess` is `RuleAgent`'s own argument and its existing draw.

In [ ]:
#| export
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache

import numpy as np

from explRL.clue_agents import ClueLiar, RuleAgent, known
from explRL.clue_colony import Q, tally
from explRL.colony_agents import ColonyAgent, MistakenAgent, faultClass, memberClass
from explRL.colony_board import InSquad, failures, stage4Members

B3_KNOBS = dict(trustOne=(0.0, 1.0), spread=(0.0, 1.0), shareAll=(0.0, 1.0), guess=(0.0, 0.5))
MIXED = ('trustOne', 'spread', 'shareAll')                 # the knobs with a mixin and the b3 generator; guess has neither


@dataclass(frozen=True)
class B3Params:
    "One value per knob (FR-A1). guess None: the cell's own value. B3Params() is θ_default; `world()` is what world['b3'] holds."
    trustOne: float = 0.0
    spread: float = 0.0
    shareAll: float = 0.0
    guess: float = None

    def __post_init__(self):
        for k, (lo, hi) in B3_KNOBS.items():
            v = getattr(self, k)
            if v is not None and not lo <= v <= hi:
                raise ValueError(f'{k} must be in [{lo}, {hi}], got {v!r}')

    def world(self):
        return {k: float(getattr(self, k)) for k in B3_KNOBS if getattr(self, k)}


def b3Rng(seed, slot):
    "The b3 generator of one member (FR-A3): its own stream, never the member's rng or the board's."
    return np.random.default_rng([int(seed), 22, int(slot)])


def isDefault(b3, world=None):
    "True when b3 (world['b3'], a dict or B3Params) changes nothing: no knob set, or each at 0 (guess: the world's own value)."
    b3 = b3.world() if isinstance(b3, B3Params) else dict(b3 or {})
    return all(not b3.get(k) for k in MIXED) and b3.get('guess') in (None, (world or {}).get('guess', 0.0))

In [ ]:
#| export
class TrustOne:
    """knownFor: the base agent's (so failFlag's flag-aware rule is kept), then for each clue absent from it the leading value
    among values with no failures, when it has fewer than q senders and this agent's draw for (k, j, value) < trustOne."""
    def knownFor(self, b):
        out, q, tal = super().knownFor(b), getattr(self, 'q', Q), tally(self.channel, b)
        fl, seen = failures(b) if getattr(b, 'failFlag', False) else {}, self.__dict__.setdefault('_trust', {})
        for k, cl in tal.items():
            for j, vs in cl.items():
                ok = [v for v in vs if not fl.get((k, j, v[0]))]
                if k in out and j not in out[k] and ok and len(ok[0][1]) < q:
                    if (key := (k, j, ok[0][0])) not in seen:
                        seen[key] = bool(self.b3rng.random() < self.b3['trustOne'])
                    if seen[key]:
                        out[k][j] = ok[0][0]
        return out


class Spread:
    """choose: the base agent's action; if it is an open and the b3 draw < spread, the open goes instead to the live subtask
    with the fewest openers (ties: most known clues, then value, then at random on the b3 generator)."""
    def choose(self, b, t):
        act = super().choose(b, t)
        if act[0] != 'open' or not self.b3rng.random() < self.b3['spread']:
            return act
        kn = self.knownFor(b) if hasattr(self, 'knownFor') else known(self.id, b, self.channel)
        live, n = [x for x in b.open if b.tasks[x].expire > t + 1] or list(b.open), Counter(b.opened)
        key = {x: (n[x] - (b.opened[self.id] == x), -len(kn[x]), -b.tasks[x].value) for x in live}
        top = [x for x in live if key[x] == min(key.values())]
        return ('open', int(top[int(self.b3rng.integers(len(top)))]))


class ShareAll:
    """_shareable(b, kn, t, *a) (GuardAgent passes its subtask order): with probability shareAll this turn, the share policy
    is 'always' for the call (any open subtask, not only one a teammate has open); restored after."""
    def _shareable(self, b, kn, t, *a):
        if not self.b3rng.random() < self.b3['shareAll']:
            return super()._shareable(b, kn, t, *a)
        old, self.share = self.share, 'always'
        try:
            return super()._shareable(b, kn, t, *a)
        finally:
            self.share = old

In [ ]:
#| export
_MIXINS = dict(trustOne=TrustOne, spread=Spread, shareAll=ShareAll)


@lru_cache(None)
def _b3Class(cls, names):
    return type('B3' + cls.__name__, tuple(_MIXINS[n] for n in names) + (cls,), {})


def b3Members(world, members, seed):
    """FR-A2: each honest numpy member (not a liar or a mistaken member; for an InSquad wrapper, its inner member) gains the
    mixins whose knob is set (trustOne only on counting-board agents, ColonyAgent and its subclasses), with b3Rng(seed, slot);
    guess sets RuleAgent's own argument. Identity, and no generator made, when isDefault. Returns `members`."""
    b3 = dict(world.get('b3') or {})
    if isDefault(b3, world):
        return members
    for slot, m in enumerate(members):
        x = m.m if isinstance(m, InSquad) else m
        if not isinstance(x, RuleAgent) or isinstance(x, (ClueLiar, MistakenAgent)):
            continue
        if b3.get('guess') is not None:
            x.guess = float(b3['guess'])
        names = tuple(n for n in MIXED if b3.get(n) and (n != 'trustOne' or isinstance(x, ColonyAgent)))
        if names:
            x.__class__, x.b3, x.b3rng = _b3Class(type(x), names), b3, b3Rng(seed, slot)
    return members

## The numpy replay with b3 members

`b3Replay` is `colony_trace.replay`'s numpy branch (the same board, channel, member generators and turn order), with `b3Members` applied after `stage4Members`. At θ_default it equals `replay` (`test_b3_defaults`).

In [ ]:
#| export
def b3Replay(cfg, seed):
    "colony_trace.replay(cfg, seed) for numpy members, with world['b3']'s behaviours -> Replay(rows, [], board, answers, nLiar, T)."
    from explRL.clue_runs import clueMetrics, clueSetup
    from explRL.colony_trace import Replay
    from explRL.llm_agent import runTurns
    from explRL.multi import spawnRngs
    T, nL, w = cfg.turnsPerAgent(), cfg.nLiar, cfg.world
    b, ch = clueSetup(cfg, seed)
    rngs = spawnRngs(seed + 1000, cfg.N)
    members = [faultClass(w)(ch, rngs[i]) if i < nL else memberClass(w, i, nL)(ch, rngs[i]) for i in range(cfg.N)]
    hist = runTurns(b3Members(w, stage4Members(w, members), seed), b, T)
    row = dict(episode=0, **clueMetrics(cfg, b, hist, range(nL, cfg.N), T))
    subs = {(s[0], s[1]) for s in b.submits}
    ans = {(float(r.t), r.agent): r.extras[0] for r in hist.records if (float(r.t), r.agent) in subs}
    return Replay([row], [], b, ans, nL, T)

## Tests

In [ ]:
# test_b3_defaults (AC-2, FR-A3, NFR-1): with world['b3'] absent or all default, numpy cells at 8, 16, 32 and 64 agents
# (plain, counting board, guards, squads; liars and clean) give rows, board logs and answers equal to replay's, and no b3Rng
import copy
from explRL.bridge_grid import ARMS20, c1Cell
from explRL.colony4 import ARMS4
from explRL.colony_grid import colonyCell
from explRL.colony_trace import replay
from explRL.fix_grid import fixCell
from explRL.llm_runs import LLMConfig

made = []
_rng = b3Rng
b3Rng = lambda seed, slot: made.append(slot) or _rng(seed, slot)

def _same(a, b):
    assert a.rows == b.rows and a.answers == b.answers and a.nLiar == b.nLiar
    assert a.board.submits == b.board.submits and a.board.shareLog == b.board.shareLog

def _cfg(c, b3=None):
    w = copy.deepcopy(c['cfg'])
    if b3 is not None:
        w['world']['b3'] = b3
    return LLMConfig.fromDict(w)

cases = [colonyCell('t', a, N, 3000, f, 'numpy', arms=ARMS4) for a, N, f in
         [('plain', 8, 'clean'), ('colony', 16, 'independent'), ('colonyGuards', 32, 'independent'), ('squads', 64, 'independent'),
          ('failFlag', 16, 'independent'), ('plain', 16, 'error')]] + [c1Cell('t', 'bridge1', 3000, runner='numpy'),
         fixCell('scaleB', 'baseline', 32, 900, runner='numpy')]
for c in cases:
    ref = replay(_cfg(c), 3000)
    for b3 in (None, {}, B3Params().world(), dict(trustOne=0.0, spread=0.0, shareAll=0.0), dict(guess=c['cfg']['world'].get('guess', 0.0))):
        _same(b3Replay(_cfg(c, b3), 3000), ref)
assert made == [] and isDefault(None) and isDefault(B3Params()) and not isDefault(dict(spread=0.1))
b3Rng = _rng
for bad in (dict(trustOne=1.5), dict(guess=0.6), dict(spread=-0.1)):
    try:
        B3Params(**bad); assert False, bad
    except ValueError:
        pass

In [ ]:
# test_b3_behaviours (AC-3, FR-A2): trustOne, spread and shareAll on hand-built boards; only the members FR-A2 names change
from explRL.clue_agents import ClueBoard, ClueLiar, RuleAgent
from explRL.fix_grid import fixCell
from explRL.clue_runs import clueSetup
from explRL.colony_agents import ColonyAgent, GuardAgent, MistakenAgent

def _board(c, b3, seed=3000):
    cfg = _cfg(c, b3)
    b, ch = clueSetup(cfg, seed)
    b.advance(0.0)
    w, nL = cfg.world, cfg.nLiar
    ms = [faultClass(w)(ch, np.random.default_rng(i)) if i < nL else memberClass(w, i, nL)(ch, np.random.default_rng(i)) for i in range(cfg.N)]
    for i, m in enumerate(ms):
        m.id = i
    return b, ch, b3Members(w, stage4Members(w, ms), seed)

col = colonyCell('t', 'colonyGuards', 16, 3000, 'independent', 'numpy', arms=ARMS4)
P14 = fixCell('scaleB', 'baseline', 8, 900, runner='numpy')      # Phase 14's plain board: blackboard, one holder per clue
# trustOne: a one-sender leading value of a clue the agent does not hold is known at 1, not at 0; the same decision every turn
for p, expect in ((1.0, True), (0.0, False)):
    b, ch, ms = _board(col, dict(trustOne=p) if p else None)
    a = next(m for m in ms if type(m).__name__.endswith('ColonyAgent'))
    k = next(k for k in b.open if 0 not in b.held(a.id, k))
    ch.post(0 if a.id else 1, k, (0, b.tasks[k].task.primes[0], 1), 0.0)
    assert [0 in a.knownFor(b)[k] for _ in range(3)] == [expect] * 3, p
b, ch, ms = _board(col, dict(trustOne=0.5))
for m in ms[:4]:
    assert type(m) is ClueLiar                                # liars unchanged
assert all(isinstance(m, TrustOne) for m in ms[4:])           # guards and colony agents gain trustOne
a = ms[8]
for k in list(b.open)[:6]:
    if 0 not in b.held(a.id, k):
        ch.post(1, k, (0, b.tasks[k].task.primes[0], 1), 0.0)
first = {k: 0 in v for k, v in a.knownFor(b).items()}
assert all({k: 0 in v for k, v in a.knownFor(b).items()} == first for _ in range(5)) and len(a._trust) > 0
# spread: an open goes to a live subtask with the fewest openers
plain = colonyCell('t', 'plain', 16, 3000, 'clean', 'numpy', arms=ARMS4)
for c in (plain, col, P14):
    b, ch, ms = _board(c, dict(spread=1.0))
    a = next(m for m in ms if isinstance(m, Spread) and not isinstance(m, GuardAgent))
    b.opened = [int(b.open[0])] * len(b.opened)
    b.opened[a.id] = -1
    act = a.choose(b, 0.0)
    if act[0] == 'open':
        n = lambda x: sum(o == x for i, o in enumerate(b.opened) if i != a.id)
        live = [x for x in b.open if b.tasks[x].expire > 1.0] or list(b.open)
        assert n(act[1]) == min(n(x) for x in live), c['level']
    else:
        assert act[0] in ('share', 'submit'), act
    assert not isinstance(a, TrustOne)                         # trustOne is not set
# shareAll: a held clue whose subtask nobody has open is shared at 1, not at 0 (share 'needed')
for c in (plain, col, P14):
    got = {}
    for p in (0.0, 1.0):
        b, ch, ms = _board(c, dict(shareAll=p) if p else None)
        a = next(m for m in ms if isinstance(m, RuleAgent) and not isinstance(m, (ClueLiar, GuardAgent)))
        b.opened = [-1] * len(b.opened)
        kn = a.knownFor(b) if hasattr(a, 'knownFor') else known(a.id, b, a.channel)
        got[p] = a._shareable(b, kn, 0.0)
    assert got[0.0] is None and got[1.0] is not None and got[1.0][0] == 'share', (c['level'], got)
    assert a.share == 'needed'                                 # restored after the call
# mistaken members and guess: error-fault members unchanged; guess sets RuleAgent's argument, with no b3 generator
b, ch, ms = _board(colonyCell('t', 'plain', 16, 3000, 'error', 'numpy', arms=ARMS4), dict(spread=1.0, guess=0.2))
assert all(type(m) is MistakenAgent for m in ms[:4]) and all(m.guess == 0.2 and isinstance(m, Spread) for m in ms[4:])
b, ch, ms = _board(plain, dict(guess=0.3))
assert all(m.guess == 0.3 and type(m) is RuleAgent and not hasattr(m, 'b3rng') for m in ms)

In [ ]:
#| slow
# regression (NFR-1, T1): θ_default reruns through b3Replay of Part 0's Stage 1 cells (N 8 and 16, q 2, clean and independent)
# and Stage 4's forecast cells (N 16, 32 and 64) equal forecast/runs.csv and forecast4/runs.csv, column for column. Seeds
# 3000-3004 here; tools/b3_run.py check reruns all 30 before the fit (T4).
import time
from explRL.bench_grid import readCSV
from explRL.colony_grid import part0Grid
from explRL.colony_mech import _diff, _key, stage4Columns
from explRL.colony_trace import colonyColumns
from explRL.llm_runs import cellRow
from explRL.train import demo_artifact_dir

t0, F = time.time(), demo_artifact_dir(None, 'multiagent/colony')
ref0 = {_key(r): r for r in readCSV(F / 'forecast/runs.csv')}
ref4 = {_key(r): r for r in readCSV(F / 'forecast4/runs.csv')}
c0 = [c for c in part0Grid(seeds=range(3000, 3005), sizes=(8, 16)) if c['level'].split('|')[1] in ('clean', 'independent') and 'q3' not in c['level']]
c4 = [colonyCell('colony4Forecast', a, N, s, 'independent', 'numpy', arms=ARMS4) for a in sorted({k.split('|')[0] for _, k, _, _ in ref4})
      for N in (16, 32, 64) for s in range(3000, 3005)]
for cells, ref, cols in ((c0, ref0, lambda cfg, rp: colonyColumns(rp)), (c4, ref4, stage4Columns)):
    for c in cells:
        cfg = LLMConfig.fromDict(c['cfg'])
        row = cellRow(c, cfg, cols(cfg, b3Replay(cfg, c['seed'])))
        assert not (bad := _diff(ref[_key(row)], row)), (_key(row), bad[:5])
print(f'{len(c0)} Part 0 and {len(c4)} Stage 4 forecast boards equal at θ_default, {time.time() - t0:.0f} s')

In [ ]:
# test_b3_mirror (AC-4, FR-A4, P22-12): on C1's numpy bridge board (64 agents, bridge1, independent liars), honest holders in
# squad s + 1 post their cross clues on the mirrors and RuleBridge members relay them; with spread = 1 bridges still relay
from explRL.bridge_agents import RuleBridge
from explRL.bridge_mech import c1Columns
for b3 in (None, dict(spread=1.0), dict(trustOne=1.0, spread=1.0)):
    c = c1Cell('t', 'bridge1', 3000, runner='numpy')
    cfg = _cfg(c, b3)
    rp = b3Replay(cfg, 3000)
    col = c1Columns(cfg, rp)
    honestMirror = [m for m in rp.board.mirrorLog if m[1] >= rp.nLiar]
    assert honestMirror and col['mirrorPostsPerHolder'] > 0 and col['bridgeRelays'] > 0, (b3, col['bridgeRelays'])
    assert {s for _, s in (r[:2] for r in rp.board.relayLog)} or col['relaysPerBridgeTurn'] > 0


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()